# Essential Operations — the twenty methods that cover most exam questions

01 Core Python · **▶ 02 Pandas** · 03 Cleaning · 04 Transformation · 05 Feature Engineering · 06 Regression · 07 Model Prep · 08 Case Studies · 09 Deployment

`02_Pandas_Essentials/02_essential_operations.ipynb`

---

### In one paragraph (no jargon)

This is the toolbox notebook. None of these methods belong to a bigger topic, but between them they answer most of the questions you'll be asked: what distinct values are in this column, how many of each, which rows match my condition, sort it, apply my own calculation to every row, find the blanks. Learn these twenty and you can improvise the rest.

### After this notebook you can

- Count and list unique values, and produce a frequency table
- Filter rows on one condition and on several combined conditions
- Apply your own function (or a lambda) down a column
- Sort by one or more columns, and pull out the top N
- Find, drop and fill missing values
- Build a pivot table and a cross-tabulation

**Assumed knowledge:** `01_series_and_dataframes.ipynb`

### What's inside

1. Unique values and frequency counts
2. Filtering rows on conditions
3. Applying your own functions
4. Sorting and ranking
5. Missing values — find, drop, fill
6. Pivot tables and cross-tabs
7. ⚡ query(), nlargest() and vectorised vs .apply()
8. Exam quick-reference

---

> **▶ Runs on its own.** The next cell is the only setup you need. It imports the
> libraries and loads the data. If the `datasets/` folder isn't where it expects,
> it rebuilds an equivalent dataset in memory so **every cell below still runs** —
> handy if you copy this single `.ipynb` somewhere else.
>
> **▶ Reading the cells.** Code comments explain *what the line does*; the text
> blocks explain *why you'd do it*. Look for these markers:
> `# WHAT:` a plain-English translation · `# WHY:` the reason it matters ·
> `# 🔧 CHANGE THIS:` the knob to turn when the exam question differs ·
> **⚡ Beyond the syllabus** = optional, higher-mark techniques.

In [1]:
# =============================================================================
# SETUP — run this cell first. It is the only cell with dependencies.
# =============================================================================
# WHAT: `import` pulls in code other people have written so we don't rewrite it.
#       The `as pd` part is a nickname, so we can type `pd` instead of `pandas`.
import pandas as pd          # tables of data (think: Excel, but programmable)
import numpy as np           # fast maths on whole columns at once
import matplotlib.pyplot as plt   # charts
import warnings

warnings.filterwarnings('ignore')          # hide version-upgrade notices, keeps output readable
pd.set_option('display.max_columns', 50)   # don't hide columns behind "..."
pd.set_option('display.width', 160)
import os

def find_datasets_folder(start=None):
    """Walk upwards from this notebook looking for the shared `datasets/` folder.

    WHY: it means the notebook works whether you opened it from its own folder,
    from the top of the notes, or from anywhere else on your machine.
    """
    here = os.path.abspath(start or os.getcwd())
    for _ in range(6):                       # look up to 6 folders up
        candidate = os.path.join(here, 'datasets')
        if os.path.isdir(candidate):
            return candidate
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    return None

def load_data(filename, rebuild=None, **read_kwargs):
    """Load `filename` from the shared datasets folder, or rebuild it in memory.

    WHAT: tries to read the real file; if it can't find it, calls `rebuild()`
          which recreates a dataset with the same columns and behaviour.
    WHY:  guarantees this notebook runs even if the CSV goes missing.
    """
    folder = find_datasets_folder()
    if folder:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            reader = pd.read_excel if filename.lower().endswith(('.xlsx', '.xls')) else pd.read_csv
            print(f"Loaded '{filename}' from {folder}")
            return reader(path, **read_kwargs)
    if rebuild is None:
        raise FileNotFoundError(f"Could not find {filename} and no fallback was supplied.")
    print(f"'{filename}' not found on disk -> rebuilding an equivalent dataset in memory.")
    return rebuild()


def rebuild_customer_data():
    """Recreate customer_data.csv (200 rows) with the same columns and behaviour."""
    rng = np.random.default_rng(42)          # fixed seed -> identical numbers every run
    n = 200
    frame = pd.DataFrame({
        'Customer_ID':     np.arange(1, n + 1),
        'Age':             rng.integers(18, 70, n).astype(float),
        'Gender':          rng.choice(['Male', 'Female'], n),
        'Income':          rng.normal(60000, 18000, n).round(-2).clip(20000, 150000),
        'Purchase_Amount': rng.gamma(4, 400, n).round(2),
        'Region':          rng.choice(['North', 'South', 'East', 'West'], n),
    })
    # the real file has a scattering of blanks — reproduce them so the cleaning code has work to do
    for col, frac in [('Age', .05), ('Income', .06), ('Purchase_Amount', .04)]:
        frame.loc[rng.choice(n, int(n * frac), replace=False), col] = np.nan
    return frame

print("Setup complete. pandas", pd.__version__, "| numpy", np.__version__)

Setup complete. pandas 3.0.2 | numpy 2.4.4


In [2]:
# A small, readable frame for the mechanics, plus the 200-row customer file for realism
df = pd.DataFrame({
    'col1': [1, 2, 3, 4],
    'col2': [444, 555, 666, 444],
    'col3': ['abc', 'def', 'ghi', 'xyz'],
})
customers = load_data('customer_data.csv', rebuild=rebuild_customer_data)
print("Toy frame:")
display(df)
print(f"Customer file: {customers.shape[0]} rows x {customers.shape[1]} columns")

Loaded 'customer_data.csv' from /home/claude/work/build/Python-BDA-Complete-Notes/datasets
Toy frame:


,col1,col2,col3
0,1,444,abc
1,2,555,def
2,3,666,ghi
3,4,444,xyz


Customer file: 200 rows x 6 columns


## 1. Unique values and frequency counts

Three methods, asked for constantly:

- `.unique()` — **which** distinct values are there? (returns an array)
- `.nunique()` — **how many** distinct values? (returns one number)
- `.value_counts()` — **how many of each**? (returns a Series, sorted biggest first)

`value_counts()` is the one to reach for first on any text column — it tells you the categories *and* spots
data-entry problems ("Male", "male", "M" showing as three separate values).

In [3]:
print("Which distinct values? .unique()   ->", df['col2'].unique())
print("How many distinct?    .nunique()  ->", df['col2'].nunique())
print("\nHow many of each?     .value_counts()")
print(df['col2'].value_counts())

print("\nOn real data — instantly shows the category balance:")
print(customers['Region'].value_counts())

# normalize=True converts counts to proportions — usually the more useful figure
print("\nAs percentages (normalize=True):")
print((customers['Region'].value_counts(normalize=True) * 100).round(1).astype(str) + '%')

# dropna=False reveals blanks that value_counts hides by default — a common exam trap
print("\nIncluding missing values (dropna=False):")
print(customers['Income'].isnull().value_counts(dropna=False).rename({False: 'has value', True: 'missing'}))

Which distinct values? .unique()   -> [444 555 666]
How many distinct?    .nunique()  -> 3

How many of each?     .value_counts()
col2
444    2
555    1
666    1
Name: count, dtype: int64

On real data — instantly shows the category balance:
Region
West     59
East     56
South    40
North    38
Name: count, dtype: int64

As percentages (normalize=True):
Region
West     30.6%
East     29.0%
South    20.7%
North    19.7%
Name: proportion, dtype: str

Including missing values (dropna=False):
Income
has value    183
missing       17
Name: count, dtype: int64


## 2. Filtering rows on conditions

A condition on a column produces a **True/False Series** (a "mask"). Put that mask inside `df[...]` and pandas
keeps only the True rows.

**The two rules that break everyone:**
1. Use `&` for AND and `|` for OR — **not** the words `and`/`or`, which fail on Series.
2. **Wrap each condition in its own brackets.** `df[(a > 2) & (b == 444)]` works; without the inner brackets
   Python's operator precedence evaluates it wrongly and you get a confusing error.

In [4]:
# One condition
print("col1 greater than 2:")
display(df[df['col1'] > 2])

# TWO conditions with & (AND) — note the brackets around EACH condition
print("col1 > 2 AND col2 == 444:")
newdf = df[(df['col1'] > 2) & (df['col2'] == 444)]
display(newdf)

# | is OR;  ~ is NOT
print("col1 > 3 OR col2 == 555:")
display(df[(df['col1'] > 3) | (df['col2'] == 555)])
print("NOT col2 == 444:")
display(df[~(df['col2'] == 444)])

col1 greater than 2:


,col1,col2,col3
2,3,666,ghi
3,4,444,xyz


col1 > 2 AND col2 == 444:


,col1,col2,col3
3,4,444,xyz


col1 > 3 OR col2 == 555:


,col1,col2,col3
1,2,555,def
3,4,444,xyz


NOT col2 == 444:


,col1,col2,col3
1,2,555,def
2,3,666,ghi


In [5]:
# The four filters you'll actually use on real data

# 1. Numeric threshold
print("Customers spending over 2000 :", len(customers[customers['Purchase_Amount'] > 2000]))

# 2. Membership — .isin() beats chaining several == with |
target_regions = ['North', 'West']                    # 🔧 CHANGE THIS: your list of allowed values
print("In North or West              :", len(customers[customers['Region'].isin(target_regions)]))

# 3. Between two values (inclusive at both ends)
print("Aged 25-40 inclusive          :", len(customers[customers['Age'].between(25, 40)]))

# 4. Text matching — .str gives you every string method, applied down the column
print("Region starting with 'S'      :", len(customers[customers['Region'].str.startswith('S')]))

# Combine them — build the mask separately when it gets long, it's far easier to debug
mask = (
    (customers['Purchase_Amount'] > 1000)
    & (customers['Region'].isin(target_regions))
    & (customers['Age'].between(25, 45))
)
print(f"\nAll three conditions together : {mask.sum()} rows")
display(customers[mask].head(3))

Customers spending over 2000 : 118
In North or West              : 97
Aged 25-40 inclusive          : 65
Region starting with 'S'      : 40

All three conditions together : 42 rows


,Customer_ID,Age,Gender,Income,Purchase_Amount,Region
1,2,45.0,Male,65000.0,1900.0,North
5,6,37.0,NaN,20000.0,3900.0,West
7,8,35.0,Female,70000.0,2200.0,West


## 3. Applying your own functions

`.apply()` runs a function on every value in a column. Use it when the calculation is too complex for plain
arithmetic — otherwise plain arithmetic is far faster (there's a timing demonstration at the end of this
notebook).

In [6]:
# A named function
def times2(x):
    return x * 2

print("apply(times2) on col1:\n", df['col1'].apply(times2).tolist())

# A lambda — same thing, defined inline
print("apply(lambda) on col1:\n", df['col1'].apply(lambda x: x * 2).tolist())

# A built-in function: len() applied to every string
print("apply(len) on col3   :\n", df['col3'].apply(len).tolist())

# .apply on a whole ROW needs axis=1 — the function receives the entire row as a Series
def summarise(row):
    return f"{row['col3']}: {row['col1']} x {row['col2']}"

print("\napply across rows (axis=1):")
print(df.apply(summarise, axis=1).tolist())

# .map() is the Series-only sibling, and it also accepts a DICTIONARY for recoding —
# the cleanest way to translate codes into labels
region_codes = {'North': 'N', 'South': 'S', 'East': 'E', 'West': 'W'}
print("\n.map with a dictionary (recoding):")
print(customers['Region'].map(region_codes).value_counts().to_string())

apply(times2) on col1:
 [2, 4, 6, 8]
apply(lambda) on col1:
 [2, 4, 6, 8]
apply(len) on col3   :
 [3, 3, 3, 3]

apply across rows (axis=1):
['abc: 1 x 444', 'def: 2 x 555', 'ghi: 3 x 666', 'xyz: 4 x 444']

.map with a dictionary (recoding):
Region
W    59
E    56
S    40
N    38


## 4. Sorting and ranking

`sort_values()` sorts by content; `sort_index()` sorts by the row labels. Neither changes the original unless
you assign the result back. `ascending=False` gives largest-first, which is what "top N" questions want.

In [7]:
print("Sorted by col2 ascending (the default):")
display(df.sort_values(by='col2'))

print("Largest first, and ties broken by col1:")
display(df.sort_values(by=['col2', 'col1'], ascending=[False, True]))

# The "top 5 customers by spend" question — the most common sorting question there is
print("\nTop 5 customers by spend:")
display(customers.sort_values('Purchase_Amount', ascending=False).head(5))

# .rank() adds a position number without reordering the rows
customers_ranked = customers.assign(
    Spend_Rank=customers['Purchase_Amount'].rank(ascending=False, method='min')
)
print("\nRank column added (method='min' gives tied rows the same rank):")
display(customers_ranked.nsmallest(4, 'Spend_Rank')[['Customer_ID', 'Purchase_Amount', 'Spend_Rank']])

# Column and index names — you'll need these when a question says "print the column names"
print("\nColumns:", list(df.columns))
print("Index  :", list(df.index))

Sorted by col2 ascending (the default):


,col1,col2,col3
0,1,444,abc
3,4,444,xyz
1,2,555,def
2,3,666,ghi


Largest first, and ties broken by col1:


,col1,col2,col3
2,3,666,ghi
1,2,555,def
0,1,444,abc
3,4,444,xyz



Top 5 customers by spend:


,Customer_ID,Age,Gender,Income,Purchase_Amount,Region
47,48,23.0,Female,65000.0,4900.0,South
187,188,44.0,Female,70000.0,4900.0,North
93,94,51.0,Male,55000.0,4900.0,North
56,57,22.0,NaN,60000.0,4800.0,North
191,192,49.0,Male,NaN,4800.0,North



Rank column added (method='min' gives tied rows the same rank):


,Customer_ID,Purchase_Amount,Spend_Rank
47,48,4900.0,1.0
93,94,4900.0,1.0
187,188,4900.0,1.0
56,57,4800.0,4.0



Columns: ['col1', 'col2', 'col3']
Index  : [0, 1, 2, 3]


## 5. Missing values — find, drop, fill

The three-step routine: **find** them (`isnull().sum()`), **decide** what they mean, then **drop** or **fill**.
Dropping loses information; filling invents it. Whichever you choose, say why — that sentence is the mark.
(Full treatment in `03_Data_Cleaning/01`.)

In [8]:
gaps = pd.DataFrame({
    'col1': [1, 2, 3, np.nan],
    'col2': [np.nan, 555, 666, 444],
    'col3': ['abc', 'def', 'ghi', 'xyz'],
})

print("Where are the blanks? .isnull() gives a True/False grid:")
display(gaps.isnull())
print("Count per column:\n", gaps.isnull().sum(), "\n")

print("dropna() — removes any row containing ANY blank (4 rows -> 2):")
display(gaps.dropna())

print("dropna(thresh=2) — keep rows with at least 2 real values:")
display(gaps.dropna(thresh=2))

print("fillna('FILL') — replace every blank with the same thing:")
display(gaps.fillna('FILL'))

print("Per-column fill — the sensible version: numbers get the median, text gets a label")
display(gaps.fillna({'col1': gaps['col1'].median(),
                     'col2': gaps['col2'].median(),
                     'col3': 'Unknown'}))

Where are the blanks? .isnull() gives a True/False grid:


,col1,col2,col3
0,False,True,False
1,False,False,False
2,False,False,False
3,True,False,False


Count per column:
 col1    1
col2    1
col3    0
dtype: int64 

dropna() — removes any row containing ANY blank (4 rows -> 2):


,col1,col2,col3
1,2.0,555.0,def
2,3.0,666.0,ghi


dropna(thresh=2) — keep rows with at least 2 real values:


,col1,col2,col3
0,1.0,NaN,abc
1,2.0,555.0,def
2,3.0,666.0,ghi
3,NaN,444.0,xyz


fillna('FILL') — replace every blank with the same thing:


,col1,col2,col3
0,1.0,FILL,abc
1,2.0,555.0,def
2,3.0,666.0,ghi
3,FILL,444.0,xyz


Per-column fill — the sensible version: numbers get the median, text gets a label


,col1,col2,col3
0,1.0,555.0,abc
1,2.0,555.0,def
2,3.0,666.0,ghi
3,2.0,444.0,xyz


## 6. Pivot tables and cross-tabulations

A pivot table reshapes long data into a grid: one variable down the side, another across the top, and an
aggregate in the middle. It's the same idea as Excel's pivot table.

- `pivot_table(values=, index=, columns=, aggfunc=)` — aggregates numbers (default `mean`)
- `pd.crosstab(a, b)` — counts how often each combination occurs

In [9]:
data = {'A': ['foo', 'foo', 'foo', 'bar', 'bar', 'bar'],
        'B': ['one', 'one', 'two', 'two', 'one', 'one'],
        'C': ['x', 'y', 'x', 'y', 'x', 'y'],
        'D': [1, 3, 2, 5, 4, 1]}
pivot_demo = pd.DataFrame(data)
display(pivot_demo)

# values = the numbers to aggregate; index = down the side; columns = across the top
print("pivot_table(values='D', index=['A','B'], columns=['C']) — mean of D in each cell:")
display(pivot_demo.pivot_table(values='D', index=['A', 'B'], columns=['C']))

print("NaN just means that combination never occurs in the data — not an error.")

,A,B,C,D
0,foo,one,x,1
1,foo,one,y,3
2,foo,two,x,2
3,bar,two,y,5
4,bar,one,x,4
5,bar,one,y,1


pivot_table(values='D', index=['A','B'], columns=['C']) — mean of D in each cell:


C          x    y
A   B            
bar one  4.0  1.0
    two  NaN  5.0
foo one  1.0  3.0
    two  2.0  NaN

NaN just means that combination never occurs in the data — not an error.


In [10]:
# On real data — average spend by region and gender, with totals
print("Average purchase by Region x Gender:")
display(customers.pivot_table(values='Purchase_Amount', index='Region', columns='Gender',
                              aggfunc='mean', margins=True, margins_name='ALL').round(0))

# Several aggregations at once
print("\nCount and mean together:")
display(customers.pivot_table(values='Purchase_Amount', index='Region',
                              aggfunc=['count', 'mean', 'sum']).round(0))

# crosstab — pure counts of each combination
print("\npd.crosstab — how many customers in each Region/Gender cell:")
display(pd.crosstab(customers['Region'], customers['Gender'], margins=True))

# normalize='index' turns each row into percentages — the version a manager wants
print("\nAs row percentages:")
display((pd.crosstab(customers['Region'], customers['Gender'], normalize='index') * 100).round(1))

Average purchase by Region x Gender:


Gender,Female,Male,ALL
Region,,,
East,2781.0,2466.0,2598.0
North,2431.0,2481.0,2456.0
South,3218.0,2511.0,2854.0
West,2491.0,2322.0,2404.0
ALL,2721.0,2440.0,2572.0



Count and mean together:


,count,mean,sum
,Purchase_Amount,Purchase_Amount,Purchase_Amount
Region,,,
East,55,2591.0,142500.0
North,37,2392.0,88500.0
South,40,2812.0,112500.0
West,54,2496.0,134800.0



pd.crosstab — how many customers in each Region/Gender cell:


Gender,Female,Male,All
Region,,,
East,21,30,51
North,16,17,33
South,17,18,35
West,23,27,50
All,77,92,169



As row percentages:


Gender,Female,Male
Region,,
East,41.2,58.8
North,48.5,51.5
South,48.6,51.4
West,46.0,54.0


### ⚡ Beyond the syllabus — `.query()`, `.nlargest()` and why `.apply()` is usually the wrong answer

Three upgrades. `.query()` writes filters as readable English. `.nlargest()` replaces sort-then-slice with one clearer, faster call. And the big one: **`.apply()` runs a Python loop behind the scenes** — on a column of numbers, plain arithmetic is often 50–100× faster. Knowing *when not to use apply* is a genuine analyst-level distinction.

In [11]:
# ---- .query() — filters that read like a sentence ---------------------------
target_regions = ['North', 'West']
classic = customers[(customers['Purchase_Amount'] > 1000) &
                    (customers['Region'].isin(target_regions)) &
                    (customers['Age'] < 45)]

# @variable_name lets you refer to Python variables inside the query string
readable = customers.query("Purchase_Amount > 1000 and Region in @target_regions and Age < 45")

print("Same rows:", len(classic) == len(readable), f"({len(readable)} rows)")
print("\nCompare the two ways of writing it:")
print("  classic : customers[(customers['Purchase_Amount'] > 1000) & (customers['Region'].isin(...))]")
print("  query   : customers.query('Purchase_Amount > 1000 and Region in @target_regions')")

# ---- .nlargest() / .nsmallest() — 'top N' without sorting the whole frame ----
print("\nTop 3 spenders, three ways — all identical:")
print("  sort+head:", customers.sort_values('Purchase_Amount', ascending=False)
                              .head(3)['Customer_ID'].tolist())
print("  nlargest :", customers.nlargest(3, 'Purchase_Amount')['Customer_ID'].tolist())
print("  Top 3 per region (groupby + nlargest):")
display(customers.dropna(subset=['Purchase_Amount'])
                 .groupby('Region', group_keys=False)
                 .apply(lambda g: g.nlargest(2, 'Purchase_Amount'), include_groups=False)
                 [['Purchase_Amount']].head(8))

Same rows:

 True (54 rows)

Compare the two ways of writing it:
  classic : customers[(customers['Purchase_Amount'] > 1000) & (customers['Region'].isin(...))]
  query   : customers.query('Purchase_Amount > 1000 and Region in @target_regions')

Top 3 spenders, three ways — all identical:
  sort+head: [48, 188, 94]
  nlargest : [48, 94, 188]
  Top 3 per region (groupby + nlargest):


,Purchase_Amount
95,4800.0
154,4600.0
93,4900.0
187,4900.0
47,4900.0
197,4600.0
111,4700.0
138,4500.0


In [12]:
# ---- Vectorised arithmetic vs .apply() — measured ---------------------------
import time
big = pd.DataFrame({'amount': np.random.default_rng(0).gamma(4, 400, 200_000)})

start = time.perf_counter(); slow = big['amount'].apply(lambda x: x * 1.18); t_apply = time.perf_counter() - start
start = time.perf_counter(); fast = big['amount'] * 1.18;                    t_vec   = time.perf_counter() - start

print(f".apply(lambda x: x * 1.18)  : {t_apply*1000:8.1f} ms")
print(f"df['amount'] * 1.18         : {t_vec*1000:8.1f} ms")
print(f"-> vectorised is {t_apply/t_vec:.0f}x faster, and identical: {slow.equals(fast)}")

print("""
WHEN TO USE WHICH
  arithmetic on numbers      ->  df['a'] * 1.18            (never .apply)
  two-way condition          ->  np.where(cond, a, b)
  many conditions            ->  np.select([c1, c2], [v1, v2], default=v3)
  banding a number           ->  pd.cut(...)
  recoding categories        ->  df['col'].map({...})
  text operations            ->  df['col'].str.upper() / .strip() / .contains()
  genuinely custom row logic ->  .apply(fn, axis=1)        <- the legitimate use""")

# The same job four ways, so you can see the translation
spend = customers['Purchase_Amount']
print("\nLabel High/Low — four equivalent routes:")
print("  np.where :", np.where(spend > 1500, 'High', 'Low')[:5])
print("  .apply   :", spend.apply(lambda x: 'High' if x > 1500 else 'Low').head(5).tolist())
print("  .map+cut :", pd.cut(spend, [-np.inf, 1500, np.inf], labels=['Low', 'High']).head(5).tolist())
print("  .mask    :", pd.Series('Low', index=spend.index).mask(spend > 1500, 'High').head(5).tolist())

.apply(lambda x: x * 1.18)  :     54.6 ms
df['amount'] * 1.18         :      1.6 ms


-> vectorised is 35x faster, and identical: True

WHEN TO USE WHICH
  arithmetic on numbers      ->  df['a'] * 1.18            (never .apply)
  two-way condition          ->  np.where(cond, a, b)
  many conditions            ->  np.select([c1, c2], [v1, v2], default=v3)
  banding a number           ->  pd.cut(...)
  recoding categories        ->  df['col'].map({...})
  text operations            ->  df['col'].str.upper() / .strip() / .contains()
  genuinely custom row logic ->  .apply(fn, axis=1)        <- the legitimate use

Label High/Low — four equivalent routes:
  np.where : ['High' 'High' 'High' 'Low' 'High']
  .apply   : ['High', 'High', 'High', 'Low', 'High']
  .map+cut : ['High', 'High', 'High', 'Low', 'High']
  .mask    : ['High', 'High', 'High', 'Low', 'High']


---

## Exam quick-reference

| To do this | Write this |
|---|---|
| Distinct values | `df['c'].unique()` |
| How many distinct | `df['c'].nunique()` |
| Frequency table | `df['c'].value_counts()` |
| As percentages | `df['c'].value_counts(normalize=True)` |
| Filter, one condition | `df[df['c'] > 2]` |
| Filter, AND / OR / NOT | `df[(a) & (b)]` · `|` · `~` |
| Value in a list | `df[df['c'].isin(['N','W'])]` |
| Between two values | `df[df['c'].between(25, 40)]` |
| Readable filter | `df.query('c > 2 and d == 444')` |
| Apply to a column | `df['c'].apply(fn)` |
| Apply across a row | `df.apply(fn, axis=1)` |
| Recode with a dictionary | `df['c'].map({'a': 1})` |
| Sort | `df.sort_values('c', ascending=False)` |
| Top N | `df.nlargest(5, 'c')` |
| Missing per column | `df.isnull().sum()` |
| Drop / fill blanks | `df.dropna()` · `df.fillna(0)` |
| Pivot table | `df.pivot_table(values=, index=, columns=, aggfunc=)` |
| Cross-tab counts | `pd.crosstab(df.a, df.b)` |

### Adapting this in the exam

- 'How many of each?' → `value_counts()`. 'Average by group?' → `groupby` (notebook 03).
- 'Top/bottom N' → `nlargest(N, 'col')` or `sort_values(...).head(N)`.
- 'Rows where…' → build a boolean mask. Long conditions: assign the mask to a variable first.
- 'Summarise A by B and C' → `pivot_table(values='A', index='B', columns='C')`.

### Traps that cost marks

- Use `&` and `|`, never `and`/`or`, with Series. The words raise `ValueError: truth value of a Series is ambiguous`.
- Every condition needs its own brackets: `df[(a > 2) & (b == 3)]`.
- `value_counts()` **excludes** missing values by default — pass `dropna=False` to see them.
- `sort_values()` returns a new frame. Without assigning it back, nothing changes.
- `.apply()` on a numeric column is slow and usually unnecessary — use arithmetic or `np.where`.
- `dropna()` removes a row if **any** column is blank. On a wide table that can delete almost everything — check `.shape` afterwards.
- In `pivot_table`, `values` must be numeric. Passing a text column returns an empty result with no error.